In [0]:
!pip install spotipy

In [0]:
%restart_python

In [0]:
from pyspark.sql import SparkSession

In [0]:
#Creamos las session de apache spark en una variable

spark = SparkSession.builder.getOrCreate()

In [0]:
# spark.stop()

In [0]:
# import spotify packages

import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

In [0]:
# create a proyect and retrieval the credentials 

# https://developer.spotify.com/documentation/web-api



In [0]:
# Create a Spotify client

client_credentials_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager)

In [0]:
# retrieval your playlist

top_tracks = sp.playlist_tracks('7EbiTIskJft6zwA8WDC4pb', limit=100) #7EbiTIskJft6zwA8WDC4pb / 37i9dQZF1DXbITWG1ZJKYt / 3VzB0idVneaUskLFv8j47y

In [0]:
import pandas as pd
import seaborn as sns

import warnings 
warnings.filterwarnings('ignore')

In [0]:
# extract from the playlist 

list_all_song = []

for track in top_tracks['items']:

    track_name = track['track']['name']
    artist_name = track['track']['artists'][0]['name']
    popularity = track['track']['popularity']
    explicit = track['track']['explicit']

    release_date = track['track']['album']['release_date']
    duration_min = track['track']['duration_ms']/60000 # convert ms to min

    list_all_song.append({'track_name':track_name, 'artist_name':artist_name, 'popularity':popularity, 'explicit': explicit,'release_date':release_date,'duration_min':duration_min})

df_all_songs = pd.DataFrame(list_all_song)

In [0]:
df_all_songs.head()

In [0]:
df_all_songs.shape

In [0]:
# cast date

df_all_songs['release_date'] = pd.to_datetime(df_all_songs['release_date'], errors='ignore')

In [0]:
def get_bracket_date(x):
    
    if x <= '2000-01-01':
        return 'Antes de los 2000'
    elif x > '2000-01-01' and x <= '2010-01-01':
        return 'Entre 2000 y 2010'
    elif x > '2010-01-01' and x <= '2020-01-01':
        return 'Entre 2010 y 2020'
    else:
        return 'Después de 2020'

In [0]:
df_all_songs['release_date_bracket'] = df_all_songs['release_date'].apply(get_bracket_date)

In [0]:
df_all_songs.head()

# Introduction to Spark

In [0]:
# create a spark dataframe

df_spark = spark.createDataFrame(df_all_songs)

In [0]:
df_spark

In [0]:
# print data type

type(df_spark)

In [0]:
df_spark.show()

In [0]:
df_spark.show(5)

In [0]:
df_spark.printSchema()

In [0]:
# select a subset of columns

df_spark.select('track_name','explicit').show()

In [0]:
from pyspark.sql.functions import col

In [0]:
df_spark.select(col("track_name"),col("explicit")).show()

In [0]:
df_spark.select('track_name','explicit').show()

In [0]:
df_spark.select(df_spark.colRegex("`^.*name*`")).show()

In [0]:
df_spark.show(5)

In [0]:
# cast column

df_spark.withColumn("popularity",col("popularity").cast("Integer")).show()

In [0]:
df_spark.printSchema()

In [0]:
df_spark = df_spark.withColumn("popularity", col("popularity").cast("integer"))

In [0]:
df_spark.printSchema()

In [0]:
df_spark.withColumn("duration_hrs",col("duration_min")/60).show(5)

In [0]:
df_spark.show(5)

In [0]:
df_spark = df_spark.withColumn("duration_hrs", col("duration_min") / 60)

In [0]:
df_spark.show(5)

In [0]:
df_spark = df_spark.withColumnRenamed("track_name","pista")

In [0]:
df_spark.show(5)

In [0]:
from pyspark.sql.functions import lit

In [0]:
df_spark = df_spark.withColumn("user", lit("joao"))

In [0]:
df_spark.show(5)

# Filter & Sort functions

In [0]:
df_spark.filter(df_spark.release_date_bracket == "Después de 2020").show(truncate=False)

In [0]:
df_spark.filter(df_spark.release_date_bracket == "Después de 2020").count()

In [0]:
df_spark.filter(df_spark.release_date_bracket != "Después de 2020").show(truncate=False) 

In [0]:
df_spark.filter(~(df_spark.release_date_bracket == "Después de 2020")).show(truncate=False)

In [0]:
df_spark.filter(col("artist_name") == "Los Nota Lokos").show(truncate=False) 

In [0]:
df_spark.filter("explicit == False").show()

In [0]:
df_spark.filter("duration_min > 3").show()

In [0]:
df_spark.filter( (df_spark.popularity >80) & (df_spark.duration_min  >= 3) ).show(truncate=False)  

In [0]:
list_date_bckt = ["Después de 2020","Entre 2000 y 2010"]

df_spark.filter(df_spark.release_date_bracket.isin(list_date_bckt)).show()

In [0]:
df_spark.filter(~df_spark.release_date_bracket.isin(list_date_bckt)).show()

In [0]:
df_spark.filter(df_spark.artist_name.like("%aga%")).show()

In [0]:
df_spark.count()

In [0]:
df_spark_distinct = df_spark.distinct()
df_spark_distinct.count(), df_spark.count() # no hay repetidos

In [0]:
df_spark_distinct = df_spark.dropDuplicates()
df_spark_distinct.count(),  df_spark.count()

In [0]:
df_spark.sort('popularity', ascending = False).show(10)

In [0]:
df_spark.sort('popularity','duration_min', ascending = False).show(10)

In [0]:
df_spark.orderBy("popularity","duration_min", ascending = False).show(10)

In [0]:
df_spark.orderBy("duration_min", ascending = False).show(10)

In [0]:
df_spark.createOrReplaceTempView("joao_playlist")

In [0]:
spark.sql("select pista,artist_name, popularity from joao_playlist ORDER BY popularity desc").show(10)

In [0]:
%sql
select pista,artist_name, popularity from joao_playlist ORDER BY popularity desc

In [0]:
df_result = spark.sql("""
    SELECT pista, artist_name, popularity 
    FROM joao_playlist
    ORDER BY popularity DESC
""")

In [0]:
df_result.show(5)

In [0]:
df_result.count()

In [0]:
%sql
select pista,artist_name, popularity from joao_playlist ORDER By popularity desc

# Agg

In [0]:
from pyspark.sql.functions import col,sum,avg,max,min,countDistinct, median

In [0]:
# check some metrics

# df_all_songs.groupby('artist_name').agg({'track_name':'nunique','popularity':'mean','duration_min':'mean','release_date':'max'}).sort_values('track_name', ascending = False).head(10)

In [0]:
df_spark.groupBy("artist_name").agg(countDistinct("pista").alias("nunique_pista"), \
         avg("popularity").alias("avg_popularity"), \
         avg("duration_min").alias("avg_duration_min"), \
         max("release_date").alias("max_release_date")
     ) \
    .sort('nunique_pista', ascending = False).show(20)

In [0]:
df_spark.groupBy('artist_name').avg('popularity').sort('avg(popularity)', ascending = False).show(10)

In [0]:
# df_all_songs.groupby('release_date_bracket').agg({'track_name':'nunique','popularity':'mean','duration_min':'mean'}).sort_values('popularity', ascending = False)

In [0]:
df_spark.groupBy('release_date_bracket').agg(countDistinct("pista").alias("nunique_pista"), \
         avg("popularity").alias("avg_popularity"), \
         avg("duration_min").alias("avg_duration_min")
     ) \
    .sort('avg_popularity', ascending = False).show(20)

In [0]:
# df_all_songs.groupby('explicit').agg({'popularity':['mean','median'],'track_name':'nunique','duration_min':['mean','median']})

In [0]:
df_spark.groupBy('explicit').agg(countDistinct("pista").alias("nunique_pista"), \
         avg("popularity").alias("avg_popularity"), \
         median("popularity").alias("median_popularity"), \
         avg("duration_min").alias("avg_duration_min"),\
        median("duration_min").alias("median_duration_min")
     ).show(20)

In [0]:
df_spark.groupBy('explicit').agg(countDistinct("pista").alias("nunique_pista"), \
         avg("popularity").alias("avg_popularity"), \
         median("popularity").alias("median_popularity"), \
         avg("duration_min").alias("avg_duration_min"),\
        median("duration_min").alias("median_duration_min")
     ).where(col("median_popularity") >= 60).show(truncate=False)

In [0]:
# df_spark.filter(df_spark.release_date_bracket == "Después de 2020").show(truncate=False)

In [0]:
df_spark.filter(df_spark.release_date_bracket == "Después de 2020").groupBy('explicit').agg(countDistinct("pista").alias("nunique_pista"), \
         avg("popularity").alias("avg_popularity"), \
         median("popularity").alias("median_popularity"), \
         avg("duration_min").alias("avg_duration_min"),\
        median("duration_min").alias("median_duration_min")
     ).show(20)